# Module 1: Setup & First LLM Call with LangChain

Welcome to the first coding lab! In this notebook, we will:
1. Initialize environment variables and set up our API keys.
2. Establish a connection to free models using **OpenRouter** and **Hugging Face**.
3. Understand how to format messages (`SystemMessage`, `HumanMessage`, `AIMessage`).
4. Invoke the model and dissect the raw response payload.
5. Learn the difference between `invoke` and `stream` for handling model responses.

### Step 1: Install & Import Dependencies

Ensure you have activated your virtual environment and installed the requirements before running this cell. Here, we'll load our API keys from the `.env` file.

In [1]:
import os
from dotenv import load_dotenv

# Load environment variables from the .env file
load_dotenv()

# Let's verify that the required environment variables are set
openrouter_api_key = os.getenv("OPENROUTER_API_KEY")
hf_api_token = os.getenv("HUGGINGFACEHUB_API_TOKEN")

print("OPENROUTER_API_KEY configured:", openrouter_api_key is not None and len(openrouter_api_key) > 0)
print("HUGGINGFACEHUB_API_TOKEN configured:", hf_api_token is not None and len(hf_api_token) > 0)

OPENROUTER_API_KEY configured: True
HUGGINGFACEHUB_API_TOKEN configured: True


---
## Connection Option A: OpenRouter (OpenAI-Compatible Wrapper)

OpenRouter uses the same API structure as OpenAI. This means we can import the `ChatOpenAI` client wrapper from `langchain_openai`, pointing it to OpenRouter's URL and passing an OpenRouter free model. 

We'll use `google/gemma-2-9b-it:free` as our default model. It is high-performing, instruction-tuned, and free to use.

In [11]:
from langchain_openai import ChatOpenAI

# Initialize the model
openrouter_model = ChatOpenAI(
    openai_api_base="https://openrouter.ai/api/v1",
    openai_api_key=openrouter_api_key,
    model_name="nvidia/nemotron-3-nano-omni-30b-a3b-reasoning:free",
    temperature=0.3, # Low temperature makes outputs more deterministic
)

print("OpenRouter ChatOpenAI model wrapper initialized successfully!")

OpenRouter ChatOpenAI model wrapper initialized successfully!


### Structuring Conversation Messages

Rather than sending a single raw string prompt, modern ChatModels expect an array of messages representing the conversational context:
- `SystemMessage`: Tells the AI model its role, personality, or bounds (e.g. "You are a concise python assistant").
- `HumanMessage`: Represents your query.

In [12]:
from langchain_core.messages import SystemMessage, HumanMessage

messages = [
    SystemMessage(content="You are an experienced software architect who explains complex topics in exactly two sentences."),
    HumanMessage(content="What is a design pattern?")
]

# We invoke the model by passing the messages list
response = openrouter_model.invoke(messages)
print("Response type:", type(response))
print("\nAI Reply Content:")
print(response.content)

Response type: <class 'langchain_core.messages.ai.AIMessage'>

AI Reply Content:
Adesign pattern is a reusable, abstract solution to a recurring software design problem that captures expert knowledge and promotes consistency. It provides a proven template that can be adapted to various contexts, improving code readability, maintainability, and development speed.


### Dissecting the AIMessage Output Payload

The output of our chat model is a sub-class of `BaseMessage` called `AIMessage`. It contains the model's text response, but also critical metadata like token counts, system parameters, and finishing reasons.

In [13]:
print("--- RAW AIMESSAGE PAYLOAD ---")
print(repr(response))

print("\n--- KEY COMPONENTS ---")
# Content generated by model
print("Content:", response.content)

# Provider response metadata (e.g. finish reason, model names)
print("\nResponse Metadata:", response.response_metadata)

# Token usage statistics
usage = response.response_metadata.get("token_usage", {})
print(f"\nUsage Summary: Prompt={usage.get('prompt_tokens', 'N/A')} | Completion={usage.get('completion_tokens', 'N/A')} | Total={usage.get('total_tokens', 'N/A')}")

--- RAW AIMESSAGE PAYLOAD ---
AIMessage(content='Adesign pattern is a reusable, abstract solution to a recurring software design problem that captures expert knowledge and promotes consistency. It provides a proven template that can be adapted to various contexts, improving code readability, maintainability, and development speed.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 228, 'prompt_tokens': 49, 'total_tokens': 277, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': 0, 'reasoning_tokens': 239, 'rejected_prediction_tokens': None, 'image_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0, 'cache_write_tokens': 0, 'video_tokens': 0}, 'cost': 0, 'is_byok': False, 'cost_details': {'upstream_inference_cost': 0, 'upstream_inference_prompt_cost': 0, 'upstream_inference_completions_cost': 0}}, 'model_provider': 'openai', 'model_name': 'nvidia/nemotron-3-nano-omni-30b-a3b-reasoning-202

---
## Connection Option B: HuggingFace Inference API (Alternative)

If you prefer to connect directly to models hosted on Hugging Face (e.g. Meta Llama 3 8B), we can use `langchain-huggingface`.

Note: We create a text generation pipeline via `HuggingFaceEndpoint` and wrap it in `ChatHuggingFace` so it accepts the list-of-messages format correctly.

In [14]:
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace

try:
    # 1. Instantiate the LLM pipeline
    hf_llm = HuggingFaceEndpoint(
        repo_id="meta-llama/Meta-Llama-3-8B-Instruct",
        huggingfacehub_api_token=hf_api_token,
        task="text-generation",
        timeout=60,
    )
    
    # 2. Wrap it with ChatHuggingFace to standardize message inputs
    hf_model = ChatHuggingFace(llm=hf_llm)
    print("HuggingFace ChatModel initialized successfully!")
    
    # Test call
    hf_response = hf_model.invoke([
        SystemMessage(content="You write code responses in JSON only."),
        HumanMessage(content="List the colors of the rainbow.")
    ])
    print("\nHuggingFace Reply:")
    print(hf_response.content)
except Exception as e:
    print("HuggingFace connection failed. Verify API token and access permissions.")
    print("Error details:", e)

HuggingFace ChatModel initialized successfully!

HuggingFace Reply:
[{"color": "Red"}, {"color": "Orange"}, {"color": "Yellow"}, {"color": "Green"}, {"color": "Blue"}, {"color": "Indigo"}, {"color": "Violet"}]


---
## Stream vs Invoke

When building chat applications, waiting for the entire response to generate before showing it to the user creates a sluggish experience. LangChain provides a `.stream()` method that yields individual token chunks as they are generated in real-time.

In [16]:
import sys

query = "Write a short paragraph explaining the philosophy of open source software."

print("--- USING INVOKE (Blocks until done) ---")
blocking_response = openrouter_model.invoke([HumanMessage(content=query)])
print(blocking_response.content)

print("\n--- USING STREAM (Real-time output) ---")
stream_chunks = openrouter_model.stream([HumanMessage(content=query)])

for chunk in stream_chunks:
    # Print each chunk content as it arrives, avoiding line buffers
    sys.stdout.write(chunk.content)
    sys.stdout.flush()
print() # Print empty line at the end

--- USING INVOKE (Blocks until done) ---
Opensource software is built on the philosophy that knowledge and technology should be freely shared, examined, and improved by anyone. It embraces transparency, collaboration, and community-driven development, allowing developers to view, modify, and distribute source code without restrictive patents or licensing fees. This openness fosters innovation, rapid problem‑solving, and collective ownership, aiming to create higher‑quality, more secure, and adaptable software that benefits users and developers alike.

--- USING STREAM (Real-time output) ---
Opensource software embodies the philosophy that knowledge and technology should be freely shared, examined, and improved by anyone. It promotes transparency, collaboration, and community-driven development, allowing developers worldwide to inspect the code, fix bugs, add features, and adapt the software to diverse needs without restrictive licenses. By democratizing access to the source code, open 

### What is in a chunk?
Each item yielded by the `.stream()` iterator is a `ChatGenerationChunk` (inheriting from `AIMessageChunk`). 
These chunks can be added together (`chunk1 + chunk2`) to accumulate the output. LangChain models implement addition operators directly on message chunks.

In [17]:
stream_chunks = openrouter_model.stream([HumanMessage(content="Count to 3.")])

chunk_list = list(stream_chunks)
print("First chunk representation:", repr(chunk_list[0]))

# Combine all chunks together to reconstruct the full AIMessage
full_message = chunk_list[0]
for next_chunk in chunk_list[1:]:
    full_message += next_chunk
    
print("\nRecombined message content:", full_message.content)

First chunk representation: AIMessageChunk(content='', additional_kwargs={}, response_metadata={'model_provider': 'openai'}, id='lc_run--019e6d01-6af2-7131-8f02-0f3bad666043', tool_calls=[], invalid_tool_calls=[], tool_call_chunks=[])

Recombined message content: 1  
2  
3
